# naja-schematic: interactive netlist schematics in a notebook

[naja-schematic](https://github.com/keplertech/naja-schematic) is an open-source schematic viewer for gate-level and RTL netlists. The viewer is C++/[Dear ImGui](https://github.com/ocornut/imgui) compiled to WebAssembly. The netlist stays in this Python kernel, loaded and edited with [najaeda](https://github.com/najaeda/naja), and the viewer only asks for the part you're looking at.

Run the cells one at a time, top to bottom (Shift+Enter). The views read the live netlist, so after *Run all* every view shows it as the last cell left it (with `u_red` renamed). It's a beta: expect rough edges, and [report them](https://github.com/keplertech/naja-schematic/issues).

In [ ]:
# Installs (or upgrades to) the released package. The notebook test sets
# NAJA_SCHEMATIC_NOTEBOOK_TEST to keep the freshly built wheel it runs against.
import os
if "NAJA_SCHEMATIC_NOTEBOOK_TEST" not in os.environ:
    %pip install -q -U naja-schematic

import najaeda, naja_schematic, anywidget
print("naja-schematic", naja_schematic.__version__, "| anywidget", anywidget.__version__)

## 1. Load a design
An RGB LED dimmer in SystemVerilog: one 8-bit counter shared by three PWM channels, each lighting its LED while the counter is below its duty cycle. najaeda elaborates it to gates and flops with [slang](https://github.com/MikePopoloski/slang), keeping the module hierarchy.

In [ ]:
%%writefile rgb_pwm.sv
// RGB LED dimmer: one free-running counter, three PWM channels.
module counter #(parameter W = 8) (
  input  logic         clk, rst,
  output logic [W-1:0] q);
  always_ff @(posedge clk)
    if (rst) q <= '0;
    else     q <= q + 1'b1;
endmodule

// The output is high while the counter is below the duty cycle.
module pwm_channel #(parameter W = 8) (
  input  logic         clk,
  input  logic [W-1:0] cnt, duty,
  output logic         out);
  always_ff @(posedge clk) out <= cnt < duty;
endmodule

module rgb_pwm (
  input  logic       clk, rst,
  input  logic [7:0] duty_r, duty_g, duty_b,
  output logic       led_r, led_g, led_b);
  logic [7:0] cnt;
  counter     u_counter(.clk, .rst, .q(cnt));
  pwm_channel u_red    (.clk, .cnt, .duty(duty_r), .out(led_r));
  pwm_channel u_green  (.clk, .cnt, .duty(duty_g), .out(led_g));
  pwm_channel u_blue   (.clk, .cnt, .duty(duty_b), .out(led_b));
endmodule

In [ ]:
from najaeda import netlist
netlist.reset()
top = netlist.load_system_verilog("rgb_pwm.sv", netlist.SystemVerilogConfig(top="rgb_pwm"))
[(inst.get_name(), inst.get_model_name()) for inst in top.get_child_instances()]

## 2. Explore it
Things to try in the viewer:
- **Tree (left):** expand `u_red`. Instances, terms and nets load on demand.
- **Trace a cone:** right-click the `led_r` term → *Trace to Driver*. The trace stops at the channel's output flop. Then right-click that flop's `D` pin in the schematic → *Trace to Driver*: the whole 8-bit comparator appears inside its `u_red` frame, reaching back to the counter and the `duty_r` inputs.
- **Click a hollow pin** in the schematic to pull in the net behind it. **Double-click a dashed box** to show all its pins.
- **Right-click** an instance or pin → *Show Properties*. Module frames come from *View > Show Hierarchy*.
- **Scroll** to zoom and **drag** to pan.

In [ ]:
view = naja_schematic.show(height=600)
view

## 3. Drive the view from Python
Point the view at an instance, by name path or with a najaeda `Instance`. It opens the tree down to the instance, selects it and draws it with all its pins open, ready to extend.

In [ ]:
view.show_instance(["u_red"])

In [ ]:
# The same with a najaeda Instance
view.show_instance(top.get_child_instance("u_counter"))

## 4. Read the selection back
Click an instance in the tree or a box in the schematic. The callback prints on each click, and the next cell gives the selection as a najaeda `Instance`.

In [ ]:
handler = view.on_select(lambda inst: print("selected:", inst))

In [ ]:
print("selected_path:", view.selected_path)
inst = view.selected
print(inst, "->", inst.get_model_name() if inst else None)

## 5. Export the schematic as SVG
`export_svg()` asks the view for what its schematic shows, as a standalone SVG: the whole sheet, with text kept as text. It's ready to drop into a doc or a PR. The view answers after this cell has run, so the SVG appears below and in `rgb_pwm.svg` once the view has drawn something. Trace the comparator cone above (section 2), then run the cell again to export it.

The viewer's own *File > Export Schematic as SVG...* menu downloads the same file.

In [ ]:
from IPython.display import SVG, Markdown, display
out = display(Markdown("*Waiting for the view to draw...*"), display_id=True)
view.export_svg("rgb_pwm.svg", lambda svg: out.update(SVG(svg)))

## 6. Open a view on one instance
A new view, started on the blue channel.

In [ ]:
naja_schematic.show(["u_blue"], height=500)

## 7. Edit the design, then view it again
Views read the live netlist, so a fresh view shows the edit: `u_led_red` instead of `u_red`. To update an existing view instead, call `view.design_changed()`.

In [ ]:
top.get_child_instance("u_red").set_name("u_led_red")
naja_schematic.show(height=500)

## 8. A real design: tinyrocket
A gate-level RISC-V core is in its own notebook, since najaeda holds one design at a time: [open the tinyrocket notebook in Colab](https://colab.research.google.com/github/keplertech/naja-schematic/blob/main/python/notebooks/tinyrocket.ipynb).

## Known rough edges
- The mouse wheel zooms the viewer, so it doesn't scroll the page while the pointer is over a view.
- Older views don't refresh after the design is edited unless you call `view.design_changed()`.
- Any errors show in the browser console (F12) or the cell output. Issues are welcome: https://github.com/keplertech/naja-schematic/issues